# ML Coding — Day 1: Transformers & friends

**~1 hour, vectorized NumPy only.** No scalar/loop versions — go straight to the vectorized solution,
and use **no Python loops over elements** in your answers. Each question is grounded in a real system
or a paper (reference + why it matters in the cell). Fill the stub, run the test, peek at the
collapsed `ref_` cells only after trying.

Today: **Q1 warmup** (embedding retrieval) · **Q2 medium** (attention) · **Q3 medium** (LayerNorm /
RMSNorm) · **Q4 hard** (multi-head causal attention) · **Q5 hard, tensor trick** (conv2d via
`sliding_window_view`).

---

## Q1 — Cosine top-k retrieval  ·  `[warmup]`

**Where it's real:** the core of vector search / RAG — embed a query, score it against a database of
embeddings by cosine similarity, return the top-k. (FAISS, ScaNN, every vector DB does this; the
brute-force version below is exactly what they optimize.)

Implement `cosine_topk(Q, D, k) -> (idx, sims)`:
- `Q` is `(n, d)` queries, `D` is `(m, d)` database vectors.
- Return `idx` `(n, k)` = indices of the top-k database vectors per query by **cosine** similarity,
  sorted most-similar-first, and `sims` `(n, k)` = their similarities.

**Vectorized plan:** L2-normalize both sides, one matmul for the full `(n, m)` score matrix, then
`np.argpartition` for an O(m) top-k followed by a small sort of the k winners. Cosine ignores
magnitude — scaling a query must not change the ranking.

In [36]:
import numpy as np


def cosine_topk(Q, D, k):
    q_size = np.sqrt(np.sum(Q**2, axis = -1, keepdims=True)) # [n]
    d_size = np.sqrt(np.sum(D**2, axis = -1, keepdims=True)) # [m]
    ranking = (Q/q_size) @ (D/d_size).T # [n, m]
    # print(ranking)
     # Unsorted top-k indices per row
    idx = np.argpartition(ranking, -k, axis=1)[:, -k:]  # [n, k]

    # Get their scores
    sims = np.take_along_axis(ranking, idx, axis=1)  # [n, k]

    # Sort top-k by similarity descending
    order = np.argsort(-sims, axis=1)
    print(order)
    idx = np.take_along_axis(idx, order, axis=1)
    sims = np.take_along_axis(sims, order, axis=1)
    # print(idx, sims)
    return idx, sims

In [37]:
# --- Q1 tests ---
def _q1():
    D = np.array([[1., 0, 0], [0, 1., 0], [0, 0, 1.], [1., 1., 0]])
    Q = np.array([[1., 0, 0], [0.9, 0.1, 0]])
    idx, sims = cosine_topk(Q, D, k=2)
    assert idx.shape == (2, 2) and sims.shape == (2, 2)
    assert idx[0, 0] == 0 and abs(sims[0, 0] - 1.0) < 1e-9      # q0 == D0 exactly
    assert np.all(sims[:, 0] >= sims[:, 1] - 1e-12)            # sorted desc
    idx2, _ = cosine_topk(Q * 100.0, D, k=2)                   # scale-invariant
    assert np.array_equal(idx2, idx)
    print("Q1 OK")

_q1()

[[1 0]
 [1 0]]
[[1 0]
 [1 0]]
Q1 OK


## Q2 — Scaled dot-product attention  ·  `[medium]`

**Paper:** Vaswani et al., *Attention Is All You Need*, NeurIPS 2017 (arXiv:1706.03762).
**Why it matters:** this single operation is the heart of every transformer. The `1/√d` scaling is
the key detail — without it, large dot products push softmax into a saturated, low-gradient regime as
`d` grows.

Implement `attention(Q, K, V, mask=None) -> (out, weights)`:
- `Q` `(..., Tq, d)`, `K` `(..., Tk, d)`, `V` `(..., Tk, dv)` (leading dims broadcast as batch).
- `scores = Q·Kᵀ / √d`; if `mask` (boolean, broadcastable to `(..., Tq, Tk)`, `True` = keep) is
  given, disallowed positions get `-inf`; softmax over the **last** axis (numerically stable);
  `out = weights · V`.

**Vectorized plan:** `np.matmul`/`np.swapaxes` for batched `Q·Kᵀ`, `np.where(mask, scores, -inf)`,
stable softmax (subtract per-row max), another matmul. No loops over batch or time.

In [ ]:
def attention(Q, K, V, mask=None):
    raise NotImplementedError

In [ ]:
# --- Q2 tests ---
def _q2():
    rng = np.random.default_rng(0)
    Q = rng.standard_normal((2, 3, 4)); K = rng.standard_normal((2, 5, 4)); V = rng.standard_normal((2, 5, 6))
    out, w = attention(Q, K, V)
    assert out.shape == (2, 3, 6) and w.shape == (2, 3, 5)
    assert np.allclose(w.sum(-1), 1.0)                         # weights are a distribution
    mask = np.ones((2, 3, 5), bool); mask[..., 4] = False
    out2, w2 = attention(Q, K, V, mask)
    assert np.allclose(w2[..., 4], 0.0) and np.allclose(w2.sum(-1), 1.0)   # masked key gets 0
    Ke = np.ones((1, 4, 4)); Qe = rng.standard_normal((1, 2, 4)); Ve = rng.standard_normal((1, 4, 6))
    _, we = attention(Qe, Ke, Ve)
    assert np.allclose(we, 1 / 4)                             # identical keys -> uniform weights
    assert out.max() <= V.max() + 1e-9 and out.min() >= V.min() - 1e-9     # convex combo of V
    print("Q2 OK")

_q2()

## Q3 — LayerNorm & RMSNorm  ·  `[medium]`

**Papers:** Ba, Kiros, Hinton, *Layer Normalization*, 2016 (arXiv:1607.06450); Zhang & Sennrich,
*Root Mean Square Layer Normalization*, NeurIPS 2019 (arXiv:1910.07467).
**Why it matters:** LayerNorm stabilizes training by normalizing each token's features to zero-mean /
unit-variance. RMSNorm drops the mean-centering and the bias, normalizing only by the root-mean-square
— cheaper and just as effective, which is why **LLaMA / T5 use RMSNorm**. Implementing both side by
side makes the difference concrete.

Implement (normalize over the **last** axis, with learnable affine params):
- `layer_norm(x, gamma, beta, eps=1e-5)` → `(x - mean) / √(var + eps) · gamma + beta`
- `rms_norm(x, gamma, eps=1e-5)` → `x / √(mean(x²) + eps) · gamma`  (no mean subtraction, no beta)

**Vectorized plan:** reductions with `axis=-1, keepdims=True` so the stats broadcast back over `x`.

In [ ]:
def layer_norm(x, gamma, beta, eps=1e-5):
    raise NotImplementedError


def rms_norm(x, gamma, eps=1e-5):
    raise NotImplementedError

In [ ]:
# --- Q3 tests ---
def _q3():
    rng = np.random.default_rng(1)
    x = rng.standard_normal((4, 8)) * 5.0 + 2.0
    g, b = np.ones(8), np.zeros(8)
    y = layer_norm(x, g, b)
    assert np.allclose(y.mean(-1), 0.0, atol=1e-6)            # zero mean per row
    assert np.allclose(y.std(-1), 1.0, atol=1e-3)            # unit std per row
    assert np.allclose(layer_norm(x, g * 2.0, b + 1.0), y * 2.0 + 1.0, atol=1e-5)   # affine
    r = rms_norm(x, g)
    assert np.allclose(np.sqrt(np.mean(r ** 2, -1)), 1.0, atol=1e-3)               # unit RMS
    assert not np.allclose(r.mean(-1), 0.0)                  # RMSNorm does NOT center the mean
    print("Q3 OK")

_q3()

## Q4 — Multi-head causal self-attention  ·  `[hard]`

**Paper:** Vaswani et al., 2017 (as Q2). **Why it matters:** splitting the model dim into `H` heads
lets the model attend to different subspaces in parallel; the **causal mask** (lower-triangular)
makes it autoregressive — the basis of GPT-style decoders. The reshape→transpose→batched-matmul
→reshape dance is the part people fumble.

Implement `multihead_attention(X, Wq, Wk, Wv, Wo, num_heads, causal=True) -> out`:
- `X` `(B, T, D)`; `Wq, Wk, Wv, Wo` each `(D, D)`; `D` divisible by `num_heads` (head dim `d = D/H`).
- Project `X` to Q/K/V, **split into heads** `(B, H, T, d)`, scaled dot-product attention per head
  with a causal mask when `causal`, **concat heads** back to `(B, T, D)`, then apply `Wo`.

**Vectorized plan:** `reshape(B, T, H, d).transpose(0, 2, 1, 3)` to split; `np.tril` for the causal
mask; batched `np.matmul` does all heads/batches at once; reverse the transpose+reshape to concat.
No loops over heads or batch.

In [ ]:
def multihead_attention(X, Wq, Wk, Wv, Wo, num_heads, causal=True):
    raise NotImplementedError

In [ ]:
# --- Q4 tests ---
def _q4():
    rng = np.random.default_rng(2)
    B, T, D, H = 2, 4, 6, 3
    d = D // H
    X = rng.standard_normal((B, T, D))
    Wq, Wk, Wv, Wo = (rng.standard_normal((D, D)) for _ in range(4))
    out = multihead_attention(X, Wq, Wk, Wv, Wo, H, causal=True)
    assert out.shape == (B, T, D)

    # oracle: compute each head with an explicit loop (loops allowed in tests)
    Q = (X @ Wq).reshape(B, T, H, d).transpose(0, 2, 1, 3)
    K = (X @ Wk).reshape(B, T, H, d).transpose(0, 2, 1, 3)
    V = (X @ Wv).reshape(B, T, H, d).transpose(0, 2, 1, 3)
    cmask = np.tril(np.ones((T, T), bool))
    heads = []
    for h in range(H):
        sc = Q[:, h] @ np.swapaxes(K[:, h], -1, -2) / np.sqrt(d)
        sc = np.where(cmask, sc, -np.inf)
        sc = sc - sc.max(-1, keepdims=True)
        wt = np.exp(sc); wt /= wt.sum(-1, keepdims=True)
        heads.append(wt @ V[:, h])
    expected = np.concatenate(heads, axis=-1) @ Wo
    assert np.allclose(out, expected, atol=1e-9)

    # causal property: token 0 attends only to itself, so future tokens can't change out[:, 0]
    X2 = X.copy(); X2[:, 1:] += 100.0
    out2 = multihead_attention(X2, Wq, Wk, Wv, Wo, H, causal=True)
    assert np.allclose(out[:, 0], out2[:, 0], atol=1e-9)
    print("Q4 OK")

_q4()

## Q5 — conv2d via `sliding_window_view`  ·  `[hard · tensor trick]`

**Where it's real:** this is **im2col** — the trick frameworks (cuDNN, Caffe) use to turn a
convolution into one big matmul (Chellapilla, Puri, Simard, *High Performance Convolutional Neural
Networks for Document Processing*, 2006). **Why it matters:** it shows a convolution *is* a structured
matrix multiply, and it's the canonical "manipulate strides/axes, then contract with einsum" exercise.

Implement `conv2d(x, w) -> y` (multi-channel 2-D **cross-correlation**, valid mode, no loops):
- `x` `(H, W, Cin)`, `w` `(Cout, kh, kw, Cin)` → `y` `(H-kh+1, W-kw+1, Cout)`.
- `y[i, j, o] = Σ_{a,b,c} x[i+a, j+b, c] · w[o, a, b, c]`.

**Vectorized plan:** `np.lib.stride_tricks.sliding_window_view(x, (kh, kw), axis=(0, 1))` gives a
patch tensor of shape `(Ho, Wo, Cin, kh, kw)` (a *view*, no copy); contract it against `w` with a
single `np.einsum`. The whole thing is one windowing call + one einsum.

In [ ]:
def conv2d(x, w):
    raise NotImplementedError

In [ ]:
# --- Q5 tests ---
def _q5():
    rng = np.random.default_rng(3)
    H, W, Cin, Cout, kh, kw = 7, 6, 3, 4, 3, 2
    x = rng.standard_normal((H, W, Cin)); w = rng.standard_normal((Cout, kh, kw, Cin))
    y = conv2d(x, w)
    Ho, Wo = H - kh + 1, W - kw + 1
    assert y.shape == (Ho, Wo, Cout)

    expected = np.zeros((Ho, Wo, Cout))                       # explicit-loop oracle
    for i in range(Ho):
        for j in range(Wo):
            patch = x[i:i + kh, j:j + kw, :]                  # (kh, kw, Cin)
            for o in range(Cout):
                expected[i, j, o] = np.sum(patch * w[o])
    assert np.allclose(y, expected, atol=1e-10)
    print("Q5 OK")

_q5()

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import numpy as np


def ref_cosine_topk(Q, D, k):
    Q = np.asarray(Q, float); D = np.asarray(D, float)
    Qn = Q / np.linalg.norm(Q, axis=1, keepdims=True)
    Dn = D / np.linalg.norm(D, axis=1, keepdims=True)
    S = Qn @ Dn.T                                   # (n, m) cosine similarities
    k = min(k, D.shape[0])
    part = np.argpartition(-S, k - 1, axis=1)[:, :k]            # top-k, unordered  (O(m))
    rows = np.arange(S.shape[0])[:, None]
    order = np.argsort(-S[rows, part], axis=1)                  # sort the k winners desc
    idx = part[rows, order]
    return idx, S[rows, idx]


def ref_attention(Q, K, V, mask=None):
    Q = np.asarray(Q, float); K = np.asarray(K, float); V = np.asarray(V, float)
    d = Q.shape[-1]
    scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(d)           # (..., Tq, Tk)
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)
    scores = scores - scores.max(axis=-1, keepdims=True)       # stable softmax
    w = np.exp(scores)
    w = w / w.sum(axis=-1, keepdims=True)
    return w @ V, w


def ref_layer_norm(x, gamma, beta, eps=1e-5):
    x = np.asarray(x, float)
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps) * gamma + beta


def ref_rms_norm(x, gamma, eps=1e-5):
    x = np.asarray(x, float)
    ms = np.mean(x ** 2, axis=-1, keepdims=True)
    return x / np.sqrt(ms + eps) * gamma


def ref_multihead_attention(X, Wq, Wk, Wv, Wo, num_heads, causal=True):
    X = np.asarray(X, float)
    B, T, D = X.shape
    H = num_heads
    d = D // H

    def split(M):                                   # (B, T, D) -> (B, H, T, d)
        return M.reshape(B, T, H, d).transpose(0, 2, 1, 3)

    Q, K, V = split(X @ Wq), split(X @ Wk), split(X @ Wv)
    scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(d)           # (B, H, T, T)
    if causal:
        scores = np.where(np.tril(np.ones((T, T), bool)), scores, -np.inf)
    scores = scores - scores.max(axis=-1, keepdims=True)
    w = np.exp(scores); w = w / w.sum(axis=-1, keepdims=True)
    ctx = w @ V                                                # (B, H, T, d)
    ctx = ctx.transpose(0, 2, 1, 3).reshape(B, T, D)           # concat heads
    return ctx @ Wo


def ref_conv2d(x, w):
    x = np.asarray(x, float); w = np.asarray(w, float)
    _Cout, kh, kw, _Cin = w.shape
    win = np.lib.stride_tricks.sliding_window_view(x, (kh, kw), axis=(0, 1))   # (Ho,Wo,Cin,kh,kw)
    return np.einsum("ijcab,oabc->ijo", win, w)


_i, _s = ref_cosine_topk(np.eye(3), np.eye(3), 1)
assert _i[:, 0].tolist() == [0, 1, 2] and np.allclose(_s[:, 0], 1.0)
_o, _w = ref_attention(np.zeros((1, 2, 4)), np.ones((1, 3, 4)), np.arange(18.).reshape(1, 3, 6))
assert np.allclose(_w, 1 / 3)
assert np.allclose(ref_layer_norm(np.array([[1., 2., 3.]]), np.ones(3), np.zeros(3)).mean(), 0.0, atol=1e-7)
assert ref_multihead_attention(np.zeros((1, 3, 4)), *[np.eye(4)] * 4, 2).shape == (1, 3, 4)
assert ref_conv2d(np.ones((4, 4, 1)), np.ones((1, 2, 2, 1)))[0, 0, 0] == 4.0
print("reference solutions OK")